In [12]:
%matplotlib qt
import numpy as np
import probeinterface as pi
from probeinterface.plotting import plot_probegroup

probegroup = pi.ProbeGroup()

# Parameters definition
num_arrays = 4
channels_per_array = 32
columns = 4
rows = 8
pitch_x = 400.0  # um
pitch_y = 400.0  # um
contact_radius = 15.0 # um
border_margin = 150.0 # um (distance from outer contacts to the physical edge)

global_channel_index = 0

for i in range(num_arrays):
    # Generate 2D positions for a 4x8 grid
    positions = []
    for r in range(rows):
        for c in range(columns):
            positions.append([c * pitch_x, r * pitch_y])
    positions = np.array(positions)

    # Initialize the single probe and define contacts
    probe = pi.Probe(ndim=2, si_units='um')
    probe.set_contacts(positions=positions, shapes='circle', shape_params={'radius': contact_radius})
    
    # Generate the physical contour of the probe
    probe.create_auto_shape(probe_type='rect', margin=border_margin)

    # Arrange the 4 arrays in a 2x2 grid to prevent overlaps and mimic cortical distribution
    x_offset = (i % 2) * 4000  # 4 mm horizontal spacing
    y_offset = (i // 2) * 6000 # 6 mm vertical spacing
    probe.move([x_offset, y_offset])

    # Map the progressive physical indices (0-31, 32-63, 64-95, 96-127)
    device_indices = np.arange(global_channel_index, global_channel_index + channels_per_array)
    probe.set_device_channel_indices(device_indices)

    probegroup.add_probe(probe)
    global_channel_index += channels_per_array

# Visual verification of the reconstructed topology
plot_probegroup(probegroup, with_device_index=True)
pi.write_probeinterface('probe_layout.json', probegroup)